# D65-FairFace7-ROI Inference (Colab)

Production-style cheek Lab from a Pansor flash/no-flash zip. **No FitSkin / demographics required.**

| Step | What |
|---|---|
| 1 | Pre-AWB demosaic → reflectance \(R_0=\sqrt{A_0\odot B_0'}\) |
| 2 | Apple Vision cheek mask (landmarks in zip) |
| 3 | Pooled `tier3_affine` RGB→XYZ + Bradford CAT **5500 K → D65** |
| 4 | FairFace-7 → `specular_tone` cheek sampling → Lab |

Cohort reference: mean ΔE₀₀ ≈ **3.63** (frozen trimmed-mean-only ≈ 5.55 with `sampling="off"`).

### How to run
1. Runtime → GPU optional (FairFace runs on CPU fine)
2. Run cells top to bottom
3. Cell 2 pulls one Participant’s zips from the shared Drive folder (mount does not see Shared-with-me)
4. Cell 3 runs one zip and shows cheek / FairFace / Lab swatch
5. Cell 4 plots the pinned n=65 cohort **by ethnicity**

Shared Pansor folder: https://drive.google.com/drive/folders/1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep

> Cell 1 seeds `pipeline/` into the clone if GitHub does not have it yet.


## 0 — Setup


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q rawpy opencv-python-headless numpy matplotlib gdown
try:
    import torch, torchvision  # noqa: F401
except ImportError:
    !pip install -q torch torchvision

import os, sys, json, base64
from pathlib import Path

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"
if os.path.isdir("Fitskin"):
    !cd Fitskin && git pull --ff-only || true
else:
    !git clone {REPO_URL}

REPO = Path("Fitskin").resolve()

# Seed deployable pipeline if not yet on the cloned GitHub tip
_pipe = REPO / "pipeline" / "d65_fairface7_roi.py"
if not _pipe.is_file():
    (REPO / "pipeline").mkdir(parents=True, exist_ok=True)
    (REPO / "pipeline" / "__init__.py").write_bytes(base64.b64decode("IiIiRGVwbG95YWJsZSBjaGFydC1mcmVlIHNraW4gY29sb3JpbWV0cnkgcGlwZWxpbmVzLiIiIgoKZnJvbSAuZDY1X2ZhaXJmYWNlN19yb2kgaW1wb3J0IEQ2NUZhaXJGYWNlN1JPSVBpcGVsaW5lCgpfX2FsbF9fID0gWyJENjVGYWlyRmFjZTdST0lQaXBlbGluZSJdCg=="))
    _pipe.write_bytes(base64.b64decode("IiIiRDY1LUZhaXJGYWNlNy1ST0kgaW5mZXJlbmNlIHBpcGVsaW5lIChkZXBsb3lhYmxlIGVudHJ5IHBvaW50KS4KCkZyb3plbiBjb2xvciBwYXRoOgogIHByZS1BV0IgZmxhc2gvbm8tZmxhc2ggcmVmbGVjdGFuY2Ug4oaSIHRpZXIzIGFmZmluZSBSR0LihpJYWVog4oaSCiAgQnJhZGZvcmQgQ0FUIDU1MDBL4oaSRDY1IOKGkiBjaGVlayBMYWIuCgpPcHRpb25hbCBST0k6CiAgRmFpckZhY2UtNyBvbiBhIGxhbmRtYXJrIGZhY2UgY3JvcCBzZWxlY3RzIHNwZWN1bGFyL3NoYWRvdyBjaGVlayBzYW1wbGluZwogIChgYHNhbXBsaW5nPSdmYWlyZmFjZTcnYGAsIGRlZmF1bHQpLiBVc2UgYGBzYW1wbGluZz0nb2ZmJ2BgIGZvciB0cmltbWVkLW1lYW4KICBvbmx5IChjbGFpbWFibGUgY29sb3JpbWV0cnkgd2l0aG91dCB0aGUgUk9JIGhldXJpc3RpYykuCgpObyBGaXRTa2luLCBkZW1vZ3JhcGhpY3MsIFNDUi1BV0IsIExhYiBjb3JyZWN0b3JzLCBvciBwZXJzb24tc3BlY2lmaWMgZ2F0ZXMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHNodXRpbAppbXBvcnQgc3lzCmltcG9ydCB0ZW1wZmlsZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIERpY3QsIE9wdGlvbmFsLCBVbmlvbgoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQppZiBzdHIoUk9PVCkgbm90IGluIHN5cy5wYXRoOgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKCmZyb20gZmxhc2hfbm9mbGFzaF9zcGVjdHJhbCBpbXBvcnQgcGxhbmNrX3h5el95MSAgIyBub3FhOiBFNDAyCmZyb20gbW9kZWxzLmZhaXJmYWNlX3JhY2UgaW1wb3J0IEZhaXJGYWNlUHJlZGljdG9yLCBmYWNlX3JnYl9jcm9wX2Zyb21fbGFuZG1hcmtzICAjIG5vcWE6IEU0MDIKZnJvbSBzY3JpcHRzLmV2YWx1YXRlX3BhbnNvcjIwX2NoYXJ0ZnJlZV9kNjUgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgYXBwbGVfZmFjZV9jaGVla19tYXNrcywKICAgIGV4dHJhY3RfemlwLAogICAgbGluZWFyX3JnYl90b19wcmV2aWV3X2JnciwKICAgIGxvYWRfYWZmaW5lLAogICAgbG9hZF9hcHBsZV9sYW5kbWFya3MsCiAgICBsb2FkX2RuZ19saW5lYXIsCiAgICBtYXRjaF9mbGFzaF9leHBvc3VyZSwKICAgIG1lYW5fbGFiX29uX21hc2ssCikKCkRFRkFVTFRfQ0FMX0RJUiA9IFJPT1QgLyAiY2FsaWJyYXRpb24iIC8gInRpZXIzX2FmZmluZSIKREVGQVVMVF9GQUlSRkFDRV9ESVIgPSBST09UIC8gImNhbGlicmF0aW9uIiAvICJmYWlyZmFjZSIKCiMgU29mdCBjYXB0dXJlIGdhdGUgKGNhbWVyYS1zZXR0aW5ncyBub3RlKQpFWFBPU1VSRV9JU09fUkVKRUNUID0gMjAwLjAKRVhQT1NVUkVfU0hVVFRFUl9SRUpFQ1RfUyA9IDEuMCAvIDYwLjAKRVhQT1NVUkVfTF9SRUpFQ1QgPSA3NS4wCgoKZGVmIGV4cG9zdXJlX2ZsYWdzKAogICAgKiwKICAgIGlzbzogT3B0aW9uYWxbZmxvYXRdID0gTm9uZSwKICAgIHNodXR0ZXJfczogT3B0aW9uYWxbZmxvYXRdID0gTm9uZSwKICAgIHBpcGVsaW5lX0w6IE9wdGlvbmFsW2Zsb2F0XSA9IE5vbmUsCikgLT4gRGljdFtzdHIsIEFueV06CiAgICAiIiJTb2Z0IHJlamVjdC9yZS1wcm9tcHQgZmxhZ3MgZnJvbSBFWElGICsgcGlwZWxpbmUgTCouIiIiCiAgICBmbGFncyA9IHsKICAgICAgICAiaXNvX2dlXzIwMCI6IEZhbHNlLAogICAgICAgICJzaHV0dGVyX2dlXzFfNjAiOiBGYWxzZSwKICAgICAgICAiTF9nZV83NSI6IEZhbHNlLAogICAgICAgICJvdXRfb2ZfYmFuZCI6IEZhbHNlLAogICAgfQogICAgaWYgaXNvIGlzIG5vdCBOb25lIGFuZCBucC5pc2Zpbml0ZShpc28pOgogICAgICAgIGZsYWdzWyJpc29fZ2VfMjAwIl0gPSBmbG9hdChpc28pID49IEVYUE9TVVJFX0lTT19SRUpFQ1QKICAgIGlmIHNodXR0ZXJfcyBpcyBub3QgTm9uZSBhbmQgbnAuaXNmaW5pdGUoc2h1dHRlcl9zKToKICAgICAgICBmbGFnc1sic2h1dHRlcl9nZV8xXzYwIl0gPSBmbG9hdChzaHV0dGVyX3MpID49IEVYUE9TVVJFX1NIVVRURVJfUkVKRUNUX1MKICAgIGlmIHBpcGVsaW5lX0wgaXMgbm90IE5vbmUgYW5kIG5wLmlzZmluaXRlKHBpcGVsaW5lX0wpOgogICAgICAgIGZsYWdzWyJMX2dlXzc1Il0gPSBmbG9hdChwaXBlbGluZV9MKSA+PSBFWFBPU1VSRV9MX1JFSkVDVAogICAgZmxhZ3NbIm91dF9vZl9iYW5kIl0gPSBib29sKAogICAgICAgIGZsYWdzWyJpc29fZ2VfMjAwIl0gb3IgZmxhZ3NbInNodXR0ZXJfZ2VfMV82MCJdIG9yIGZsYWdzWyJMX2dlXzc1Il0KICAgICkKICAgIHJldHVybiBmbGFncwoKCkBkYXRhY2xhc3MKY2xhc3MgRDY1RmFpckZhY2U3Uk9JUGlwZWxpbmU6CiAgICAiIiJMb2FkLW9uY2UgYWZmaW5lICsgRmFpckZhY2UtNyBydW5uZXIgZm9yIGNoYXJ0LWZyZWUgY2hlZWsgTGFiLiIiIgoKICAgIE06IG5wLm5kYXJyYXkKICAgIGZhaXJmYWNlOiBPcHRpb25hbFtGYWlyRmFjZVByZWRpY3Rvcl0KICAgIGZpeGVkX2NhdF9rOiBmbG9hdCA9IDU1MDAuMAogICAgaGFsZl9zaXplOiBib29sID0gVHJ1ZQogICAgY2F0X2RlZ3JlZTogZmxvYXQgPSAxLjAKICAgIHNhbXBsaW5nOiBzdHIgPSAiZmFpcmZhY2U3IiAgIyBmYWlyZmFjZTcgfCBvZmYKICAgIGNhbF9kaXI6IE9wdGlvbmFsW1BhdGhdID0gTm9uZQogICAgZmFpcmZhY2VfZGlyOiBPcHRpb25hbFtQYXRoXSA9IE5vbmUKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBmcm9tX2RlZmF1bHRzKAogICAgICAgIGNscywKICAgICAgICAqLAogICAgICAgIGNhbF9kaXI6IE9wdGlvbmFsW1BhdGhdID0gTm9uZSwKICAgICAgICBmYWlyZmFjZV9kaXI6IE9wdGlvbmFsW1BhdGhdID0gTm9uZSwKICAgICAgICBmaXhlZF9jYXRfazogZmxvYXQgPSA1NTAwLjAsCiAgICAgICAgaGFsZl9zaXplOiBib29sID0gVHJ1ZSwKICAgICAgICBzYW1wbGluZzogc3RyID0gImZhaXJmYWNlNyIsCiAgICAgICAgY2F0X2RlZ3JlZTogZmxvYXQgPSAxLjAsCiAgICApIC0+ICJENjVGYWlyRmFjZTdST0lQaXBlbGluZSI6CiAgICAgICAgY2FsID0gUGF0aChjYWxfZGlyIG9yIERFRkFVTFRfQ0FMX0RJUikuZXhwYW5kdXNlcigpLnJlc29sdmUoKQogICAgICAgIGZmX2RpciA9IFBhdGgoZmFpcmZhY2VfZGlyIG9yIERFRkFVTFRfRkFJUkZBQ0VfRElSKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICAgICAgTSA9IGxvYWRfYWZmaW5lKGNhbCkKICAgICAgICBmYWlyZmFjZTogT3B0aW9uYWxbRmFpckZhY2VQcmVkaWN0b3JdID0gTm9uZQogICAgICAgIGlmIHNhbXBsaW5nIGluICgiZmFpcmZhY2UiLCAiZmFpcmZhY2U3IiwgImZhaXJmYWNlNCIpOgogICAgICAgICAgICBtb2RlID0gIjQiIGlmIHNhbXBsaW5nID09ICJmYWlyZmFjZTQiIGVsc2UgIjciCiAgICAgICAgICAgIGZhaXJmYWNlID0gRmFpckZhY2VQcmVkaWN0b3IubG9hZChtb2RlPW1vZGUsIHdlaWdodHNfZGlyPWZmX2RpcikKICAgICAgICByZXR1cm4gY2xzKAogICAgICAgICAgICBNPU0sCiAgICAgICAgICAgIGZhaXJmYWNlPWZhaXJmYWNlLAogICAgICAgICAgICBmaXhlZF9jYXRfaz1mbG9hdChmaXhlZF9jYXRfayksCiAgICAgICAgICAgIGhhbGZfc2l6ZT1ib29sKGhhbGZfc2l6ZSksCiAgICAgICAgICAgIGNhdF9kZWdyZWU9ZmxvYXQoY2F0X2RlZ3JlZSksCiAgICAgICAgICAgIHNhbXBsaW5nPXN0cihzYW1wbGluZyksCiAgICAgICAgICAgIGNhbF9kaXI9Y2FsLAogICAgICAgICAgICBmYWlyZmFjZV9kaXI9ZmZfZGlyLAogICAgICAgICkKCiAgICBkZWYgcnVuX3ppcCgKICAgICAgICBzZWxmLAogICAgICAgIHppcF9wYXRoOiBVbmlvbltzdHIsIFBhdGhdLAogICAgICAgICosCiAgICAgICAgd29ya19kaXI6IE9wdGlvbmFsW1BhdGhdID0gTm9uZSwKICAgICAgICBrZWVwX2V4dHJhY3Q6IGJvb2wgPSBGYWxzZSwKICAgICkgLT4gRGljdFtzdHIsIEFueV06CiAgICAgICAgIiIiUnVuIG9uIGEgUGFuc29yLXN0eWxlIGZsYXNoL25vLWZsYXNoIHppcCB3aXRoIEFwcGxlIGxhbmRtYXJrcy4iIiIKICAgICAgICB6aXBfcGF0aCA9IFBhdGgoemlwX3BhdGgpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgICAgICBpZiB3b3JrX2RpciBpcyBOb25lOgogICAgICAgICAgICB0bXAgPSBQYXRoKHRlbXBmaWxlLm1rZHRlbXAocHJlZml4PSJkNjVfZmY3XyIpKQogICAgICAgICAgICBjbGVhbnVwID0gbm90IGtlZXBfZXh0cmFjdAogICAgICAgICAgICBleHRyYWN0X2RpciA9IHRtcCAvIHppcF9wYXRoLnN0ZW0KICAgICAgICBlbHNlOgogICAgICAgICAgICBleHRyYWN0X2RpciA9IFBhdGgod29ya19kaXIpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkgLyB6aXBfcGF0aC5zdGVtCiAgICAgICAgICAgIGNsZWFudXAgPSBGYWxzZQogICAgICAgIHRyeToKICAgICAgICAgICAgbmYsIGZsLCBsbSA9IGV4dHJhY3RfemlwKHppcF9wYXRoLCBleHRyYWN0X2RpcikKICAgICAgICAgICAgb3V0ID0gc2VsZi5ydW5fZmlsZXMobmYsIGZsLCBsbSkKICAgICAgICAgICAgb3V0WyJ6aXBfcGF0aCJdID0gc3RyKHppcF9wYXRoKQogICAgICAgICAgICBvdXRbInppcF9zdGVtIl0gPSB6aXBfcGF0aC5zdGVtCiAgICAgICAgICAgICMgUHJlZmVyIEVYSUYgZnJvbSB6aXAgbWVtYmVyIChzYW1lIGJ5dGVzIGFzIGV4dHJhY3RlZCBuby1mbGFzaCkuCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGZyb20gc2NyaXB0cy5kbmdfZXhpZiBpbXBvcnQgcmVhZF9ub2ZsYXNoX2V4cG9zdXJlX2Zyb21femlwCgogICAgICAgICAgICAgICAgZXhpZiA9IHJlYWRfbm9mbGFzaF9leHBvc3VyZV9mcm9tX3ppcCh6aXBfcGF0aCkKICAgICAgICAgICAgICAgIG91dFsiZXhwb3N1cmUiXSA9IHsKICAgICAgICAgICAgICAgICAgICAiaXNvIjogZXhpZi5nZXQoImlzbyIpLAogICAgICAgICAgICAgICAgICAgICJzaHV0dGVyX3MiOiBleGlmLmdldCgic2h1dHRlcl9zIiksCiAgICAgICAgICAgICAgICAgICAgInNodXR0ZXJfcmF3IjogZXhpZi5nZXQoInNodXR0ZXJfcmF3IiksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICBvdXRbImV4cG9zdXJlX2ZsYWdzIl0gPSBleHBvc3VyZV9mbGFncygKICAgICAgICAgICAgICAgICAgICBpc289ZXhpZi5nZXQoImlzbyIpLAogICAgICAgICAgICAgICAgICAgIHNodXR0ZXJfcz1leGlmLmdldCgic2h1dHRlcl9zIiksCiAgICAgICAgICAgICAgICAgICAgcGlwZWxpbmVfTD1vdXQuZ2V0KCJMIiksCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICAgICAgb3V0WyJleHBvc3VyZV9lcnJvciJdID0gc3RyKGV4YykKICAgICAgICAgICAgICAgIG91dFsiZXhwb3N1cmVfZmxhZ3MiXSA9IGV4cG9zdXJlX2ZsYWdzKHBpcGVsaW5lX0w9b3V0LmdldCgiTCIpKQogICAgICAgICAgICByZXR1cm4gb3V0CiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaWYgY2xlYW51cCBhbmQgd29ya19kaXIgaXMgTm9uZToKICAgICAgICAgICAgICAgIHNodXRpbC5ybXRyZWUodG1wLCBpZ25vcmVfZXJyb3JzPVRydWUpCgogICAgZGVmIHJ1bl9maWxlcygKICAgICAgICBzZWxmLAogICAgICAgIG5vZmxhc2hfZG5nOiBVbmlvbltzdHIsIFBhdGhdLAogICAgICAgIGZsYXNoX2RuZzogVW5pb25bc3RyLCBQYXRoXSwKICAgICAgICBsYW5kbWFya3NfanNvbjogVW5pb25bc3RyLCBQYXRoXSwKICAgICkgLT4gRGljdFtzdHIsIEFueV06CiAgICAgICAgIiIiUnVuIG9uIGV4cGxpY2l0IG5vLWZsYXNoIERORywgZmxhc2ggRE5HLCBhbmQgQXBwbGUgbGFuZG1hcmsgSlNPTi4iIiIKICAgICAgICBuZiA9IFBhdGgobm9mbGFzaF9kbmcpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgICAgICBmbCA9IFBhdGgoZmxhc2hfZG5nKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICAgICAgbG1fcGF0aCA9IFBhdGgobGFuZG1hcmtzX2pzb24pLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKCiAgICAgICAgQTAgPSBsb2FkX2RuZ19saW5lYXIobmYsIGhhbGZfc2l6ZT1zZWxmLmhhbGZfc2l6ZSwgdXNlX2NhbWVyYV93Yj1GYWxzZSkKICAgICAgICBCMCA9IGxvYWRfZG5nX2xpbmVhcihmbCwgaGFsZl9zaXplPXNlbGYuaGFsZl9zaXplLCB1c2VfY2FtZXJhX3diPUZhbHNlKQogICAgICAgIGlmIEIwLnNoYXBlICE9IEEwLnNoYXBlOgogICAgICAgICAgICBCMCA9IGN2Mi5yZXNpemUoQjAsIChBMC5zaGFwZVsxXSwgQTAuc2hhcGVbMF0pLCBpbnRlcnBvbGF0aW9uPWN2Mi5JTlRFUl9BUkVBKQoKICAgICAgICBsbSA9IGxvYWRfYXBwbGVfbGFuZG1hcmtzKGxtX3BhdGgpCiAgICAgICAgXywgY2hlZWsgPSBhcHBsZV9mYWNlX2NoZWVrX21hc2tzKGxtLCBBMC5zaGFwZVswXSwgQTAuc2hhcGVbMV0pCiAgICAgICAgbl9jaGVlayA9IGludChucC5jb3VudF9ub256ZXJvKGNoZWVrKSkKICAgICAgICBpZiBuX2NoZWVrIDwgNTA6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImVtcHR5IGNoZWVrIG1hc2sgKHtuX2NoZWVrfSBweCkiKQoKICAgICAgICBmYWlyZmFjZV9tZXRhOiBEaWN0W3N0ciwgQW55XSA9IHt9CiAgICAgICAgc2FtcGxpbmdfbW9kZSA9ICJvZmYiCiAgICAgICAgZXRobmljaXR5X2Zvcl9zYW1wbGluZzogT3B0aW9uYWxbc3RyXSA9IE5vbmUKICAgICAgICBpZiBzZWxmLnNhbXBsaW5nIGluICgiZmFpcmZhY2UiLCAiZmFpcmZhY2U3IiwgImZhaXJmYWNlNCIpOgogICAgICAgICAgICBpZiBzZWxmLmZhaXJmYWNlIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkZhaXJGYWNlIHNhbXBsaW5nIHJlcXVlc3RlZCBidXQgcHJlZGljdG9yIG5vdCBsb2FkZWQiKQogICAgICAgICAgICBwcmV2aWV3ID0gbGluZWFyX3JnYl90b19wcmV2aWV3X2JncihBMCkKICAgICAgICAgICAgZmFjZV9yZ2IgPSBmYWNlX3JnYl9jcm9wX2Zyb21fbGFuZG1hcmtzKHByZXZpZXcsIGxtLCBwYWRkaW5nPTAuMzUpCiAgICAgICAgICAgIGZmID0gc2VsZi5mYWlyZmFjZS5wcmVkaWN0X3JnYihmYWNlX3JnYikKICAgICAgICAgICAgZXRobmljaXR5X2Zvcl9zYW1wbGluZyA9IGZmWyJwcmVkaWN0ZWRfZXRobmljaXR5Il0KICAgICAgICAgICAgc2FtcGxpbmdfbW9kZSA9ICJzcGVjdWxhcl90b25lIgogICAgICAgICAgICBmYWlyZmFjZV9tZXRhID0gewogICAgICAgICAgICAgICAgInByZWRpY3RlZF9ldGhuaWNpdHkiOiBmZlsicHJlZGljdGVkX2V0aG5pY2l0eSJdLAogICAgICAgICAgICAgICAgImZhaXJmYWNlX2xhYmVsIjogZmZbImZhaXJmYWNlX2xhYmVsIl0sCiAgICAgICAgICAgICAgICAiZmFpcmZhY2VfY29uZmlkZW5jZSI6IGZsb2F0KGZmWyJjb25maWRlbmNlIl0pLAogICAgICAgICAgICAgICAgImZhaXJmYWNlX21vZGUiOiBmZlsibW9kZSJdLAogICAgICAgICAgICAgICAgImZhaXJmYWNlX3Byb2JzIjoge2s6IGZsb2F0KHYpIGZvciBrLCB2IGluIChmZi5nZXQoInJhY2VfcHJvYnMiKSBvciB7fSkuaXRlbXMoKX0sCiAgICAgICAgICAgIH0KCiAgICAgICAgQjBtLCBmbGFzaF9zY2FsZTAgPSBtYXRjaF9mbGFzaF9leHBvc3VyZShBMCwgQjAsIGNoZWVrKQogICAgICAgIFIwID0gbnAuc3FydChucC5tYXhpbXVtKEEwLCAwKSAqIG5wLm1heGltdW0oQjBtLCAwKSArIDFlLTgpCgogICAgICAgIHh5el93aGl0ZSA9IHBsYW5ja194eXpfeTEoZmxvYXQoc2VsZi5maXhlZF9jYXRfayksIDAuMCkKICAgICAgICBMYWIsIHNhbXBsZV9tZXRhID0gbWVhbl9sYWJfb25fbWFzaygKICAgICAgICAgICAgUjAsCiAgICAgICAgICAgIGNoZWVrLAogICAgICAgICAgICBzZWxmLk0sCiAgICAgICAgICAgIHh5el9zY2VuZV93aGl0ZT14eXpfd2hpdGUsCiAgICAgICAgICAgIGNhdF9kZWdyZWU9ZmxvYXQoc2VsZi5jYXRfZGVncmVlKSwKICAgICAgICAgICAgbF9zYW1wbGluZz1zYW1wbGluZ19tb2RlLAogICAgICAgICAgICBldGhuaWNpdHk9ZXRobmljaXR5X2Zvcl9zYW1wbGluZywKICAgICAgICApCgogICAgICAgIG91dDogRGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgICAgICJMIjogZmxvYXQoTGFiWzBdKSwKICAgICAgICAgICAgImEiOiBmbG9hdChMYWJbMV0pLAogICAgICAgICAgICAiYiI6IGZsb2F0KExhYlsyXSksCiAgICAgICAgICAgICJuX2NoZWVrIjogbl9jaGVlaywKICAgICAgICAgICAgImZsYXNoX3NjYWxlIjogZmxvYXQoZmxhc2hfc2NhbGUwKSwKICAgICAgICAgICAgInNoYXBlIjogW2ludCh4KSBmb3IgeCBpbiBBMC5zaGFwZV0sCiAgICAgICAgICAgICJzY3JfbW9kZSI6ICJwcmVhd2JfY2F0IiwKICAgICAgICAgICAgImZpeGVkX2NhdF9rIjogZmxvYXQoc2VsZi5maXhlZF9jYXRfayksCiAgICAgICAgICAgICJjYXRfZGVncmVlIjogZmxvYXQoc2VsZi5jYXRfZGVncmVlKSwKICAgICAgICAgICAgImhhbGZfc2l6ZSI6IGJvb2woc2VsZi5oYWxmX3NpemUpLAogICAgICAgICAgICAibF9zYW1wbGluZyI6IHNlbGYuc2FtcGxpbmcsCiAgICAgICAgICAgICJjYWxfZGlyIjogc3RyKHNlbGYuY2FsX2RpcikgaWYgc2VsZi5jYWxfZGlyIGVsc2UgTm9uZSwKICAgICAgICAgICAgImxfcGVyY2VudGlsZSI6IHNhbXBsZV9tZXRhLmdldCgibF9wZXJjZW50aWxlIiksCiAgICAgICAgICAgICJpbmRpYW5fYnJhbmNoIjogc2FtcGxlX21ldGEuZ2V0KCJpbmRpYW5fYnJhbmNoIiksCiAgICAgICAgICAgICJhc2lhbl9icmFuY2giOiBzYW1wbGVfbWV0YS5nZXQoImFzaWFuX2JyYW5jaCIpLAogICAgICAgICAgICAiaXJhbmlhbl9icmFuY2giOiBzYW1wbGVfbWV0YS5nZXQoImlyYW5pYW5fYnJhbmNoIiksCiAgICAgICAgICAgICJ3aGl0ZV9icmFuY2giOiBzYW1wbGVfbWV0YS5nZXQoIndoaXRlX2JyYW5jaCIpLAogICAgICAgICAgICAibm9mbGFzaF9kbmciOiBzdHIobmYpLAogICAgICAgICAgICAiZmxhc2hfZG5nIjogc3RyKGZsKSwKICAgICAgICAgICAgImxhbmRtYXJrc19qc29uIjogc3RyKGxtX3BhdGgpLAogICAgICAgIH0KICAgICAgICBvdXQudXBkYXRlKGZhaXJmYWNlX21ldGEpCiAgICAgICAgaWYgc2FtcGxlX21ldGEuZ2V0KCJwcmVkaWN0ZWRfZXRobmljaXR5Iik6CiAgICAgICAgICAgIG91dFsicHJlZGljdGVkX2V0aG5pY2l0eSJdID0gc2FtcGxlX21ldGFbInByZWRpY3RlZF9ldGhuaWNpdHkiXQoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyb20gc2NyaXB0cy5kbmdfZXhpZiBpbXBvcnQgcmVhZF9kbmdfZXhwb3N1cmUKCiAgICAgICAgICAgIGV4aWYgPSByZWFkX2RuZ19leHBvc3VyZShuZikKICAgICAgICAgICAgb3V0WyJleHBvc3VyZSJdID0gewogICAgICAgICAgICAgICAgImlzbyI6IGV4aWYuZ2V0KCJpc28iKSwKICAgICAgICAgICAgICAgICJzaHV0dGVyX3MiOiBleGlmLmdldCgic2h1dHRlcl9zIiksCiAgICAgICAgICAgICAgICAic2h1dHRlcl9yYXciOiBleGlmLmdldCgic2h1dHRlcl9yYXciKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBvdXRbImV4cG9zdXJlX2ZsYWdzIl0gPSBleHBvc3VyZV9mbGFncygKICAgICAgICAgICAgICAgIGlzbz1leGlmLmdldCgiaXNvIiksCiAgICAgICAgICAgICAgICBzaHV0dGVyX3M9ZXhpZi5nZXQoInNodXR0ZXJfcyIpLAogICAgICAgICAgICAgICAgcGlwZWxpbmVfTD1vdXRbIkwiXSwKICAgICAgICAgICAgKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICBvdXRbImV4cG9zdXJlX2Vycm9yIl0gPSBzdHIoZXhjKQogICAgICAgICAgICBvdXRbImV4cG9zdXJlX2ZsYWdzIl0gPSBleHBvc3VyZV9mbGFncyhwaXBlbGluZV9MPW91dFsiTCJdKQoKICAgICAgICByZXR1cm4gb3V0CgoKZGVmIHJlc3VsdF90b19qc29uYWJsZShyZXN1bHQ6IERpY3Rbc3RyLCBBbnldKSAtPiBEaWN0W3N0ciwgQW55XToKICAgICIiIkVuc3VyZSBKU09OLXNlcmlhbGl6YWJsZSBjb3B5IChudW1weSBzY2FsYXJzIOKGkiBQeXRob24pLiIiIgoKICAgIGRlZiBfY29udih2OiBBbnkpIC0+IEFueToKICAgICAgICBpZiBpc2luc3RhbmNlKHYsIGRpY3QpOgogICAgICAgICAgICByZXR1cm4ge3N0cihrKTogX2NvbnYoeCkgZm9yIGssIHggaW4gdi5pdGVtcygpfQogICAgICAgIGlmIGlzaW5zdGFuY2UodiwgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgICAgIHJldHVybiBbX2NvbnYoeCkgZm9yIHggaW4gdl0KICAgICAgICBpZiBpc2luc3RhbmNlKHYsIChucC5mbG9hdGluZywgbnAuaW50ZWdlcikpOgogICAgICAgICAgICByZXR1cm4gdi5pdGVtKCkKICAgICAgICBpZiBpc2luc3RhbmNlKHYsIG5wLm5kYXJyYXkpOgogICAgICAgICAgICByZXR1cm4gdi50b2xpc3QoKQogICAgICAgIGlmIGlzaW5zdGFuY2UodiwgUGF0aCk6CiAgICAgICAgICAgIHJldHVybiBzdHIodikKICAgICAgICByZXR1cm4gdgoKICAgIHJldHVybiBfY29udihyZXN1bHQpCgoKZGVmIHdyaXRlX3Jlc3VsdF9qc29uKHJlc3VsdDogRGljdFtzdHIsIEFueV0sIHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0X3RvX2pzb25hYmxlKHJlc3VsdCksIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCg=="))
    print("Seeded pipeline/ into cloned Fitskin (not yet on remote).")
else:
    print("Found pipeline module:", _pipe)

from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

import numpy as np
import torch
from pipeline.d65_fairface7_roi import (
    D65FairFace7ROIPipeline,
    result_to_jsonable,
    write_result_json,
)

CAL_DIR = REPO / "calibration" / "tier3_affine"
FAIRFACE_DIR = REPO / "calibration" / "fairface"
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)

PANSOR_ROOT = Path("/content/Pansor Dataset")
PANSOR_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("/content/d65_fairface7_roi_inference")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"
assert FF7.is_file()

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("Setup OK.")


## 1 — Download one Participant’s zips (Drive API)

Colab’s Drive **mount does not see “Shared with me”**. This cell copies by folder ID into `/content/Pansor Dataset`.

Set `LIMIT_PARTICIPANTS = 1` for a fast smoke run. Or skip this cell and upload a zip with Cell 2b.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Drive API: copy Participant zips → /content/Pansor Dataset
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

PANSOR_FOLDER_ID = "1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep"
LIMIT_PARTICIPANTS = 1  # 1 = fast demo; None = all

drive_svc = build("drive", "v3")
FOLDER_MIME = "application/vnd.google-apps.folder"

def list_children(folder_id):
    out, token = [], None
    while True:
        resp = drive_svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=token, pageSize=1000,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        out.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return out

def download_file(file_id, dest: Path):
    if dest.is_file() and dest.stat().st_size > 0:
        print("exists", dest.name)
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = drive_svc.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            status, done = dl.next_chunk()
    print("saved", dest)

meta = drive_svc.files().get(fileId=PANSOR_FOLDER_ID, fields="name", supportsAllDrives=True).execute()
print("Shared folder:", meta["name"])

children = list_children(PANSOR_FOLDER_ID)
nested = [
    f for f in children
    if f["mimeType"] == FOLDER_MIME and f["name"].strip().lower() == "pansor dataset"
]
if nested:
    children = list_children(nested[0]["id"])
    print("Entered nested Pansor Dataset")

parts = sorted(
    [f for f in children if f["mimeType"] == FOLDER_MIME and f["name"].startswith("Participant")],
    key=lambda x: x["name"],
)
assert parts, "No Participant folders — wrong Google account or folder ID?"
todo = parts if not LIMIT_PARTICIPANTS else parts[:LIMIT_PARTICIPANTS]

for pf in todo:
    zips = [f for f in list_children(pf["id"]) if f["name"].lower().endswith(".zip")]
    indoor = [
        z for z in zips
        if "bag" not in z["name"].lower()
        and "outside" not in z["name"].lower()
        and "light" not in z["name"].lower()
    ]
    use = indoor or zips
    print(f"{pf['name']}: downloading {len(use)} zip(s)")
    for z in use:
        download_file(z["id"], PANSOR_ROOT / pf["name"] / z["name"])

print("Zips on disk:", list(PANSOR_ROOT.glob("Participant */*.zip")))


### Optional — upload a zip instead of Drive


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2b — OPTIONAL: upload one capture.zip (skip if Cell 2 already ran)
# ══════════════════════════════════════════════════════════════════════════════
# from google.colab import files
# uploaded = files.upload()  # pick a Pansor-style zip
# UPLOAD_DIR = Path("/content/uploads")
# UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
# for name, data in uploaded.items():
#     (UPLOAD_DIR / name).write_bytes(data)
#     print("saved", UPLOAD_DIR / name)


## 2 — Load pipeline once + run one zip

Shows cheek mask overlay, FairFace crop, and a Lab swatch for the trial.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Inference + visual diagnostics
# ══════════════════════════════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt
from scripts.evaluate_pansor20_chartfree_d65 import (
    extract_zip,
    load_dng_linear,
    load_apple_landmarks,
    apple_face_cheek_masks,
    linear_rgb_to_preview_bgr,
)
from models.fairface_race import face_rgb_crop_from_landmarks

SAMPLING = "fairface7"  # or "off" for frozen trimmed-mean only (~5.55 path)

pipe = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    fixed_cat_k=5500.0,
    half_size=True,
    sampling=SAMPLING,
)
print(f"Loaded pipeline  sampling={SAMPLING}  CAT=5500K  half_size=True")

zips = sorted(PANSOR_ROOT.glob("Participant */*.zip"))
if not zips:
    zips = sorted(Path("/content/uploads").glob("*.zip")) if Path("/content/uploads").is_dir() else []
assert zips, "No zips — run Cell 2 (Drive) or Cell 2b (upload)"

ZIP_PATH = zips[0]
# ZIP_PATH = next(p for p in zips if "Shuyi" in p.name)  # optional pick
print("Running:", ZIP_PATH)

result = pipe.run_zip(ZIP_PATH)
out_path = OUT_DIR / f"{ZIP_PATH.stem}.json"
write_result_json(result, out_path)

print(
    f"Lab=({result['L']:.2f}, {result['a']:.2f}, {result['b']:.2f})  "
    f"n_cheek={result.get('n_cheek')}  "
    f"FF={result.get('fairface_label')}→{result.get('predicted_ethnicity')}  "
    f"conf={result.get('fairface_confidence')}"
)
ef = result.get("exposure_flags") or {}
if ef.get("out_of_band"):
    print("exposure_flags:", ef)
else:
    print("exposure_flags: OK", ef)
print("Wrote", out_path)

# ── visuals (same zip, for display only) ──────────────────────────────────────
work = OUT_DIR / "_viz"
nf, fl, lm_path = extract_zip(ZIP_PATH, work / ZIP_PATH.stem)
A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
lm = load_apple_landmarks(lm_path)
_, cheek = apple_face_cheek_masks(lm, A0.shape[0], A0.shape[1])
preview = linear_rgb_to_preview_bgr(A0)
overlay = preview.copy()
overlay[cheek > 0] = (0.55 * overlay[cheek > 0] + 0.45 * np.array([0, 220, 80])).astype(np.uint8)
face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)

# Lab → approximate sRGB swatch for display
def lab_to_srgb_u8(L, a, b):
    # D65 XYZ via CIE Lab, then rough sRGB
    fy = (L + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0
    eps, kappa = 216 / 24389, 24389 / 27
    def f_inv(t):
        return t**3 if t**3 > eps else (116 * t - 16) / kappa
    X = 0.95047 * f_inv(fx)
    Y = 1.00000 * f_inv(fy)
    Z = 1.08883 * f_inv(fz)
    M = np.array([
        [3.2406, -1.5372, -0.4986],
        [-0.9689, 1.8758, 0.0415],
        [0.0557, -0.2040, 1.0570],
    ])
    rgb = M @ np.array([X, Y, Z])
    def lin2s(u):
        return 12.92 * u if u <= 0.0031308 else 1.055 * (max(u, 0) ** (1 / 2.4)) - 0.055
    srgb = np.clip([lin2s(float(c)) for c in rgb], 0, 1)
    return (srgb * 255).astype(np.uint8)

swatch = np.full((180, 180, 3), lab_to_srgb_u8(result["L"], result["a"], result["b"]), dtype=np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
ax[0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
ax[0].set_title("No-flash preview"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"Apple cheek (n={result['n_cheek']})"); ax[1].axis("off")
ax[2].imshow(face_rgb)
ff = result.get("fairface_label") or "?"
eth = result.get("predicted_ethnicity") or "?"
conf = result.get("fairface_confidence")
conf_s = f"{conf:.2f}" if conf is not None else "?"
ax[2].set_title(f"FairFace: {ff}\n→ {eth} ({conf_s})"); ax[2].axis("off")
ax[3].imshow(swatch)
ax[3].set_title(f"Cheek Lab\n({result['L']:.1f}, {result['a']:.1f}, {result['b']:.1f})")
ax[3].axis("off")
plt.suptitle(f"{ZIP_PATH.name}  ·  D65-FairFace7-ROI", fontsize=12)
plt.tight_layout()
plt.show()


## 3 — Cohort plots by demographic (ethnicity)

Pinned full Pansor-20 FairFace-7 cohort (**n=65**, mean ΔE₀₀ ≈ **3.63**).  
Does not require downloading every zip in Colab.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Plots by ethnicity (pinned n=65 cohort)
# ══════════════════════════════════════════════════════════════════════════════
import csv
from statistics import mean, median
import matplotlib.pyplot as plt

colors_map = {
    "Black": "#2c3e50", "Indian": "#c0392b", "Asian": "#2980b9",
    "Iranian": "#16a085", "White": "#d4a017",
}

PINNED_COHORT_ROWS = json.loads("""[{"subject_id":"P1_T1","ethnicity":"Black","de00":2.469199248916524,"pipeline_L":30.521,"pipeline_a":8.4774,"pipeline_b":12.0608},{"subject_id":"P1_T2","ethnicity":"Black","de00":2.850518823663074,"pipeline_L":31.1669,"pipeline_a":8.4397,"pipeline_b":12.3376},{"subject_id":"P1_T3","ethnicity":"Black","de00":2.414851890453143,"pipeline_L":30.2278,"pipeline_a":8.0797,"pipeline_b":11.66},{"subject_id":"P1_T4","ethnicity":"Black","de00":3.1775362832466465,"pipeline_L":31.5089,"pipeline_a":8.0923,"pipeline_b":12.0428},{"subject_id":"P2_T1","ethnicity":"White","de00":5.657487303692072,"pipeline_L":66.6278,"pipeline_a":19.1735,"pipeline_b":18.3512},{"subject_id":"P2_T2","ethnicity":"White","de00":4.81722698460258,"pipeline_L":65.6819,"pipeline_a":18.5013,"pipeline_b":18.2368},{"subject_id":"P2_T3","ethnicity":"White","de00":3.7358075897433016,"pipeline_L":64.2943,"pipeline_a":17.7366,"pipeline_b":17.7845},{"subject_id":"P2_T4","ethnicity":"White","de00":3.0199026781939824,"pipeline_L":63.4537,"pipeline_a":17.1115,"pipeline_b":17.5471},{"subject_id":"P3_T1","ethnicity":"Asian","de00":3.059202495678017,"pipeline_L":54.0755,"pipeline_a":13.3861,"pipeline_b":15.4002},{"subject_id":"P3_T2","ethnicity":"Asian","de00":2.650225990802394,"pipeline_L":56.3836,"pipeline_a":13.958,"pipeline_b":16.2278},{"subject_id":"P3_T3","ethnicity":"Asian","de00":2.6406566924952535,"pipeline_L":55.1952,"pipeline_a":13.4475,"pipeline_b":15.6197},{"subject_id":"P4_T1","ethnicity":"White","de00":4.470393076255969,"pipeline_L":59.8786,"pipeline_a":11.3306,"pipeline_b":19.7889},{"subject_id":"P4_T2","ethnicity":"White","de00":6.819530468296391,"pipeline_L":56.8653,"pipeline_a":10.9563,"pipeline_b":18.0308},{"subject_id":"P4_T3","ethnicity":"White","de00":4.642525415665498,"pipeline_L":59.3663,"pipeline_a":11.5437,"pipeline_b":18.6495},{"subject_id":"P5_T1","ethnicity":"Indian","de00":4.531370384141985,"pipeline_L":56.7907,"pipeline_a":11.1636,"pipeline_b":23.5055},{"subject_id":"P5_T2","ethnicity":"Indian","de00":4.0543812396352275,"pipeline_L":58.1168,"pipeline_a":11.5073,"pipeline_b":23.8865},{"subject_id":"P5_T3","ethnicity":"Indian","de00":1.4365077011872587,"pipeline_L":59.9283,"pipeline_a":10.471,"pipeline_b":18.8109},{"subject_id":"P5_T4","ethnicity":"Indian","de00":5.028941807749552,"pipeline_L":55.095,"pipeline_a":11.9458,"pipeline_b":22.3669},{"subject_id":"P6_T1","ethnicity":"Asian","de00":2.410683417059653,"pipeline_L":60.7408,"pipeline_a":11.2464,"pipeline_b":23.0972},{"subject_id":"P6_T2","ethnicity":"Asian","de00":1.630173595927751,"pipeline_L":59.9248,"pipeline_a":11.5362,"pipeline_b":22.5788},{"subject_id":"P6_T3","ethnicity":"Asian","de00":2.3176876117796463,"pipeline_L":61.0799,"pipeline_a":11.8621,"pipeline_b":23.0662},{"subject_id":"P7_T1","ethnicity":"Indian","de00":2.5727487788760675,"pipeline_L":51.903,"pipeline_a":13.6129,"pipeline_b":26.5187},{"subject_id":"P7_T2","ethnicity":"Indian","de00":3.4887781592722713,"pipeline_L":50.0117,"pipeline_a":12.4013,"pipeline_b":26.287},{"subject_id":"P7_T3","ethnicity":"Indian","de00":2.886962564235973,"pipeline_L":51.3622,"pipeline_a":13.5475,"pipeline_b":26.8553},{"subject_id":"P7_T4","ethnicity":"Indian","de00":2.667480354006689,"pipeline_L":51.4863,"pipeline_a":13.125,"pipeline_b":26.6298},{"subject_id":"P8_T1","ethnicity":"Indian","de00":9.246671566021673,"pipeline_L":59.1967,"pipeline_a":13.3213,"pipeline_b":19.5616},{"subject_id":"P8_T2","ethnicity":"Indian","de00":7.926403025045148,"pipeline_L":57.7384,"pipeline_a":11.9416,"pipeline_b":19.4153},{"subject_id":"P8_T3","ethnicity":"Indian","de00":9.064055201356624,"pipeline_L":58.8666,"pipeline_a":13.3617,"pipeline_b":18.7325},{"subject_id":"P9_T1","ethnicity":"Black","de00":3.5621517473173787,"pipeline_L":28.3551,"pipeline_a":8.5499,"pipeline_b":10.2869},{"subject_id":"P9_T2","ethnicity":"Black","de00":4.682535691370595,"pipeline_L":28.2056,"pipeline_a":7.0457,"pipeline_b":9.0099},{"subject_id":"P9_T3","ethnicity":"Black","de00":4.262351348828818,"pipeline_L":27.9138,"pipeline_a":7.8413,"pipeline_b":9.47},{"subject_id":"P10_T1","ethnicity":"Asian","de00":2.864263762950884,"pipeline_L":65.8882,"pipeline_a":12.8711,"pipeline_b":17.3209},{"subject_id":"P10_T2","ethnicity":"Asian","de00":2.6065775473610584,"pipeline_L":65.8871,"pipeline_a":12.4339,"pipeline_b":17.1319},{"subject_id":"P10_T3","ethnicity":"Asian","de00":2.461518656557393,"pipeline_L":65.6618,"pipeline_a":11.905,"pipeline_b":16.4402},{"subject_id":"P10_T4","ethnicity":"Asian","de00":2.801929860865549,"pipeline_L":64.9756,"pipeline_a":11.6494,"pipeline_b":16.4874},{"subject_id":"P11_T1","ethnicity":"Indian","de00":2.320364092129346,"pipeline_L":52.2545,"pipeline_a":12.2048,"pipeline_b":23.9846},{"subject_id":"P11_T2","ethnicity":"Indian","de00":3.7491169689261143,"pipeline_L":53.8952,"pipeline_a":12.7192,"pipeline_b":24.5795},{"subject_id":"P11_T3","ethnicity":"Indian","de00":3.8612689260674293,"pipeline_L":53.9726,"pipeline_a":12.6008,"pipeline_b":24.7025},{"subject_id":"P11_T4","ethnicity":"Indian","de00":2.9751212493985375,"pipeline_L":53.164,"pipeline_a":12.7703,"pipeline_b":23.8994},{"subject_id":"P12_T1","ethnicity":"Iranian","de00":1.9396378088246904,"pipeline_L":58.1829,"pipeline_a":14.5791,"pipeline_b":20.0832},{"subject_id":"P12_T2","ethnicity":"Iranian","de00":2.705757138298531,"pipeline_L":57.8394,"pipeline_a":15.0523,"pipeline_b":19.0199},{"subject_id":"P12_T3","ethnicity":"Iranian","de00":3.6120864189460584,"pipeline_L":62.1931,"pipeline_a":14.1907,"pipeline_b":18.4566},{"subject_id":"P12_T4","ethnicity":"Iranian","de00":4.470583219978213,"pipeline_L":63.9462,"pipeline_a":12.4615,"pipeline_b":18.5779},{"subject_id":"P13_T1","ethnicity":"White","de00":3.6286271769521834,"pipeline_L":59.254,"pipeline_a":11.0529,"pipeline_b":18.9443},{"subject_id":"P13_T2","ethnicity":"White","de00":3.3907120100417116,"pipeline_L":59.3923,"pipeline_a":11.1506,"pipeline_b":18.627},{"subject_id":"P14_T1","ethnicity":"Indian","de00":2.9530824522775148,"pipeline_L":45.5871,"pipeline_a":12.0651,"pipeline_b":23.47},{"subject_id":"P14_T2","ethnicity":"Indian","de00":2.672838160473732,"pipeline_L":45.9357,"pipeline_a":11.7295,"pipeline_b":23.3358},{"subject_id":"P14_T3","ethnicity":"Indian","de00":2.760027088768008,"pipeline_L":45.6544,"pipeline_a":12.1569,"pipeline_b":22.3863},{"subject_id":"P15_T1","ethnicity":"Iranian","de00":4.434614044531903,"pipeline_L":53.7136,"pipeline_a":12.9922,"pipeline_b":21.7146},{"subject_id":"P15_T2","ethnicity":"Iranian","de00":3.650044970266614,"pipeline_L":54.605,"pipeline_a":12.8725,"pipeline_b":21.5849},{"subject_id":"P16_T2","ethnicity":"White","de00":1.7785477535478762,"pipeline_L":61.3454,"pipeline_a":12.9237,"pipeline_b":16.463},{"subject_id":"P16_T3","ethnicity":"White","de00":1.8515193798609582,"pipeline_L":61.1285,"pipeline_a":12.3373,"pipeline_b":16.2436},{"subject_id":"P16_T4","ethnicity":"White","de00":3.1973361473478863,"pipeline_L":59.7268,"pipeline_a":12.7443,"pipeline_b":15.7203},{"subject_id":"P17_T1","ethnicity":"Black","de00":3.027367779628844,"pipeline_L":38.4111,"pipeline_a":11.7666,"pipeline_b":23.5083},{"subject_id":"P17_T2","ethnicity":"Black","de00":3.4855889539138603,"pipeline_L":38.4826,"pipeline_a":13.5005,"pipeline_b":25.8687},{"subject_id":"P17_T3","ethnicity":"Black","de00":2.451255333237627,"pipeline_L":41.189,"pipeline_a":14.2895,"pipeline_b":26.3068},{"subject_id":"P18_T2","ethnicity":"White","de00":3.9529572676694267,"pipeline_L":61.8964,"pipeline_a":14.7984,"pipeline_b":14.7476},{"subject_id":"P18_T3","ethnicity":"White","de00":4.981281925803477,"pipeline_L":61.1885,"pipeline_a":15.5929,"pipeline_b":14.1896},{"subject_id":"P19_T1","ethnicity":"Black","de00":3.44441197025351,"pipeline_L":31.7474,"pipeline_a":8.1321,"pipeline_b":10.9247},{"subject_id":"P19_T2","ethnicity":"Black","de00":3.586898640200647,"pipeline_L":31.9655,"pipeline_a":8.1427,"pipeline_b":10.809},{"subject_id":"P19_T3","ethnicity":"Black","de00":3.459634573041362,"pipeline_L":31.9253,"pipeline_a":8.2543,"pipeline_b":10.9776},{"subject_id":"P19_T4","ethnicity":"Black","de00":3.8878056220208377,"pipeline_L":31.8497,"pipeline_a":7.825,"pipeline_b":10.3006},{"subject_id":"P20_T1","ethnicity":"Asian","de00":4.266870341615896,"pipeline_L":61.5174,"pipeline_a":10.1401,"pipeline_b":27.301},{"subject_id":"P20_T2","ethnicity":"Asian","de00":4.308198000993366,"pipeline_L":61.4519,"pipeline_a":10.146,"pipeline_b":27.4435},{"subject_id":"P20_T3","ethnicity":"Asian","de00":4.143957517255399,"pipeline_L":61.3244,"pipeline_a":10.1904,"pipeline_b":27.2055}]""")
PINNED_SUMMARY = json.loads("""{"n":65,"mean_de00":3.6289,"median_de00":3.4444,"l_sampling":"fairface7","scr_mode":"preawb_cat","fixed_cat_k":5500.0,"white_Lp":70.0,"claimable_color_path_mean_de00":5.55,"rejected_as_overfit":["latino_to_indian_gate","white_Lp85","white_high_a_reject","shadow_bright_L_gate","black_b_gated_hiC"],"history":{"frozen_trimmed_mean":5.55,"fairface7_roi_general":3.6289,"overfit_patches_peak":2.7257},"by_ethnicity":{"Asian":{"n":13,"mean":2.9355,"median":2.6502},"Black":{"n":14,"mean":3.3402,"median":3.452},"Indian":{"n":18,"mean":4.122,"median":3.2319},"Iranian":{"n":6,"mean":3.4688,"median":3.6311},"White":{"n":14,"mean":3.996,"median":3.8444}},"note":"Generalizable stack only: frozen preAWB+5500 CAT + FairFace-7 routed coarse specular/shadow ROI (Black L_p10, White dark L_p70, Indian/Asian a*/chroma branches). Person-specific gates removed."}""")

cohort_csv = REPO / "figures" / "pansor20_fairface7" / "cohort_de00.csv"
cohort_json_path = REPO / "figures" / "pansor20_fairface7" / "cohort_de00.json"
if cohort_csv.is_file():
    cohort_rows = list(csv.DictReader(cohort_csv.open()))
    for r in cohort_rows:
        r["de00"] = float(r["de00"])
        r["pipeline_L"] = float(r.get("pipeline_L") or r.get("L") or 0)
        r["pipeline_a"] = float(r.get("pipeline_a") or r.get("a") or 0)
        r["pipeline_b"] = float(r.get("pipeline_b") or r.get("b") or 0)
    src_note = str(cohort_csv)
elif cohort_json_path.is_file():
    cohort_rows = json.loads(cohort_json_path.read_text())
    src_note = str(cohort_json_path)
else:
    cohort_rows = PINNED_COHORT_ROWS
    src_note = "embedded PINNED_COHORT_ROWS"

print(f"Cohort source: {src_note}")
print(f"n={len(cohort_rows)}  mean ΔE00≈{PINNED_SUMMARY.get('mean_de00', float('nan')):.2f}")

by_eth = {}
for r in cohort_rows:
    by_eth.setdefault(str(r.get("ethnicity") or "?").strip(), []).append(r)
order = [e for e in ["Black", "Indian", "Asian", "Iranian", "White"] if e in by_eth]
order += [e for e in sorted(by_eth) if e not in order]

# 1) ΔE histograms by ethnicity
ncols = 3
nrows = int(np.ceil(max(len(order), 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(11, 3.2 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
all_de = [float(r["de00"]) for r in cohort_rows]
vmax = max(all_de)
bins = np.arange(0, max(16.5, np.ceil(vmax) + 1.5), 1.0)
for i, eth in enumerate(order):
    ax = axes[i]
    v = [float(r["de00"]) for r in by_eth[eth]]
    ax.hist(v, bins=bins, color=colors_map.get(eth, "#7f8c8d"), edgecolor="white", alpha=0.9)
    ax.axvline(median(v), color="#e74c3c", ls="--", lw=1.4, label=f"med={median(v):.2f}")
    ax.axvline(5.55, color="#27ae60", ls=":", alpha=0.8, label="frozen 5.55")
    ax.set_title(f"{eth} (n={len(v)})")
    ax.set_xlabel(r"$\Delta E_{00}$")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8, frameon=False)
for j in range(len(order), len(axes)):
    axes[j].axis("off")
fig.suptitle(r"D65-FairFace7-ROI — $\Delta E_{00}$ by ethnicity (n=65)", fontsize=13)
plt.show()

# 2) Mean / median bars
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(order))
w = 0.35
means = [mean([float(r["de00"]) for r in by_eth[e]]) for e in order]
meds = [median([float(r["de00"]) for r in by_eth[e]]) for e in order]
ax.bar(x - w / 2, means, w, label="mean", color="#34495e")
ax.bar(x + w / 2, meds, w, label="median", color="#e67e22")
ax.axhline(5.55, color="#27ae60", ls="--", label="frozen 5.55")
ax.axhline(3.63, color="#c0392b", ls=":", label="cohort mean 3.63")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylabel(r"$\Delta E_{00}$")
ax.set_title("Mean / median ΔE00 by ethnicity")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

# 3) Cheek Lab a*–b* by ethnicity (pipeline outputs)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for eth in order:
    aa = [float(r["pipeline_a"]) for r in by_eth[eth]]
    bb = [float(r["pipeline_b"]) for r in by_eth[eth]]
    ax.scatter(aa, bb, s=42, alpha=0.85, c=colors_map.get(eth, "#7f8c8d"), label=f"{eth} (n={len(aa)})")
# mark this Colab trial if Cell 3 ran
if "result" in globals() and result.get("a") is not None:
    ax.scatter([result["a"]], [result["b"]], s=160, marker="*", c="black",
               label=f"this zip → {result.get('predicted_ethnicity')}", zorder=5)
ax.set_xlabel("a*")
ax.set_ylabel("b*")
ax.set_title("Pipeline cheek Lab (a*, b*) by ethnicity")
ax.legend(fontsize=8, frameon=False, loc="best")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# 4) L* by ethnicity box-ish strip
fig, ax = plt.subplots(figsize=(8, 4))
data_L = [[float(r["pipeline_L"]) for r in by_eth[e]] for e in order]
bp = ax.boxplot(data_L, labels=order, patch_artist=True)
for patch, eth in zip(bp["boxes"], order):
    patch.set_facecolor(colors_map.get(eth, "#7f8c8d"))
    patch.set_alpha(0.7)
ax.set_ylabel("L*")
ax.set_title("Pipeline cheek L* by ethnicity")
plt.tight_layout()
plt.show()

print(f"{'Ethnicity':10s} {'n':>4s} {'median':>8s} {'mean':>8s}")
for eth in order:
    v = [float(r["de00"]) for r in by_eth[eth]]
    print(f"{eth:10s} {len(v):4d} {median(v):8.2f} {mean(v):8.2f}")
print(f"{'ALL':10s} {len(all_de):4d} {median(all_de):8.2f} {mean(all_de):8.2f}")


## 4 — Optional batch over downloaded zips


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — OPTIONAL batch (reuse pipe from Cell 3)
# ══════════════════════════════════════════════════════════════════════════════
RUN_BATCH = False  # set True to process every zip under PANSOR_ROOT

if RUN_BATCH:
    batch_zips = sorted(PANSOR_ROOT.rglob("*.zip"))
    summary = []
    for i, zp in enumerate(batch_zips, 1):
        try:
            r = pipe.run_zip(zp)
        except Exception as exc:
            print(f"[{i:02d}/{len(batch_zips)}] FAIL {zp.name}: {exc}")
            summary.append({"zip": str(zp), "error": str(exc)})
            continue
        out = OUT_DIR / f"{zp.stem}.json"
        write_result_json(r, out)
        print(
            f"[{i:02d}/{len(batch_zips)}] {zp.name:40s}  "
            f"Lab=({r['L']:.1f},{r['a']:.1f},{r['b']:.1f})  "
            f"FF={r.get('fairface_label')}→{r.get('predicted_ethnicity')}"
        )
        summary.append({
            "zip": str(zp),
            "out": str(out),
            "L": r["L"], "a": r["a"], "b": r["b"],
            "fairface_label": r.get("fairface_label"),
            "predicted_ethnicity": r.get("predicted_ethnicity"),
            "exposure_flags": r.get("exposure_flags"),
        })
    summary_path = OUT_DIR / "summary.json"
    summary_path.write_text(
        json.dumps(result_to_jsonable({"n": len(summary), "trials": summary}), indent=2) + "\n"
    )
    print("Wrote", summary_path, "n=", len(summary))
else:
    print("Batch skipped (set RUN_BATCH = True).")


## Reference

| Path | Mean ΔE₀₀ |
|---|---:|
| Frozen trimmed mean (`sampling="off"`) | ~5.55 |
| **D65-FairFace7-ROI** (default) | **~3.63** |

Library: `from pipeline.d65_fairface7_roi import D65FairFace7ROIPipeline`  
CLI (local): `python scripts/run_d65_fairface7_roi.py --zip capture.zip --out out.json`
